In [ ]:
# v48b — KTF3: P1 Mirror Correlation — BOSS DR12 — AXIS-CORRECTED
#
# CORRECTION vs v48:
# v48 used phi(x,y,z) = (-x,y,z) where x = RA=0 direction.
# This is WRONG. The KTF3 preferred axis is at (l=277, b=-31),
# not aligned with RA=0.
#
# v48b uses the CORRECT mirror: reflection through the plane
# perpendicular to the KTF3 axis (l=277, b=-31).
# phi(r) = r - 2*(r.n_hat)*n_hat  where n_hat = KTF3 axis unit vector
#
# PRE-REGISTERED: March 2026
# Cotting, S. (2026) — academia.edu/AndrewCotting
# GitHub: github.com/Andrew-Cot/KTF3-Analyse

In [ ]:
# CELL 1 -- Imports
!pip install numpy matplotlib scipy astropy -q
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
import os, warnings
warnings.filterwarnings('ignore')

cosmo = FlatLambdaCDM(H0=67.4, Om0=0.315)

# KTF3 preferred axis in galactic coordinates
L_KTF3 = 277.0  # degrees
B_KTF3 = -31.0  # degrees

# Convert to Cartesian unit vector
l_rad = np.radians(L_KTF3)
b_rad = np.radians(B_KTF3)
n_hat = np.array([
    np.cos(b_rad) * np.cos(l_rad),
    np.cos(b_rad) * np.sin(l_rad),
    np.sin(b_rad)
])

print('v48b -- KTF3 P1 Mirror Correlation: BOSS DR12 (AXIS-CORRECTED)')
print('PRE-REGISTERED March 2026 -- Cotting, S.')
print('='*60)
print()
print('KTF3 axis: l =', L_KTF3, 'deg, b =', B_KTF3, 'deg')
print('n_hat =', np.round(n_hat, 4))
print()
print('Mirror operation: phi(r) = r - 2*(r.n_hat)*n_hat')
print('This reflects through the plane perpendicular to KTF3 axis.')
print()
print('Prediction: mirror cross-correlation peaks at r = T1/2 = 830 Mpc')

In [ ]:
# CELL 2 -- Load BOSS DR12
BOSS_FILE = 'boss_dr12_lowz.fits'
BOSS_URL  = 'https://data.sdss.org/sas/dr12/boss/lss/galaxy_DR12v5_LOWZ_North.fits.gz'

if not os.path.exists(BOSS_FILE):
    print('Downloading BOSS DR12 LOWZ North...')
    os.system('wget -q -O boss_dr12_lowz.fits.gz "' + BOSS_URL + '"')
    os.system('gunzip boss_dr12_lowz.fits.gz')
    print('Done.')
else:
    print('File present:', BOSS_FILE)

hdul = fits.open(BOSS_FILE)
data = hdul[1].data
print('Objects:', len(data))

In [ ]:
# CELL 3 -- Prepare catalog in GALACTIC coordinates
# IMPORTANT: Use galactic coordinates so KTF3 axis is meaningful

ra_eq  = data['RA'].astype(float)
dec_eq = data['DEC'].astype(float)
z      = data['Z'].astype(float)

# Convert RA/Dec to Galactic l,b
coords = SkyCoord(ra=ra_eq*u.deg, dec=dec_eq*u.deg, frame='icrs')
gal    = coords.galactic
l_deg  = gal.l.deg
b_deg  = gal.b.deg

# Quality cuts
good = (z > 0.01) & (z < 0.8) & np.isfinite(z)
l_c, b_c, z_c = l_deg[good], b_deg[good], z[good]

print('After cuts:', len(l_c), 'galaxies')
print('z range:', round(z_c.min(),3), 'to', round(z_c.max(),3))

# Convert galactic to Cartesian (same frame as n_hat)
chi    = cosmo.comoving_distance(z_c).value
l_rad  = np.radians(l_c)
b_rad  = np.radians(b_c)
x = chi * np.cos(b_rad) * np.cos(l_rad)
y = chi * np.cos(b_rad) * np.sin(l_rad)
z_cart = chi * np.sin(b_rad)
pos = np.stack([x, y, z_cart], axis=1)

# KTF3-CORRECT mirror: reflect through plane perp to KTF3 axis
# phi(r) = r - 2*(r.n_hat)*n_hat
proj = pos @ n_hat  # shape (N,)
pos_mirror = pos - 2 * np.outer(proj, n_hat)

print('Mirror axis: l =', L_KTF3, 'b =', B_KTF3)
print('Comoving range:', round(chi.min()), 'to', round(chi.max()), 'Mpc')

In [ ]:
# CELL 4 -- Mirror cross-correlation
T1     = 1660
r_pred = T1 / 2  # 830 Mpc

bins = np.arange(200, 2001, 100)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
N_BINS = len(bin_centers)

N_PAIRS = 500000
N_gal = len(pos)
rng = np.random.default_rng(42)
idx1 = rng.integers(0, N_gal, N_PAIRS)
idx2 = rng.integers(0, N_gal, N_PAIRS)
keep = idx1 != idx2
idx1, idx2 = idx1[keep], idx2[keep]

r_std    = np.linalg.norm(pos[idx2] - pos[idx1], axis=1)
r_mirror = np.linalg.norm(pos[idx2] - pos_mirror[idx1], axis=1)

N_total  = len(idx1)
N_std    = np.zeros(N_BINS)
N_mirror = np.zeros(N_BINS)
for b in range(N_BINS):
    N_std[b]    = ((r_std    >= bins[b]) & (r_std    < bins[b+1])).sum()
    N_mirror[b] = ((r_mirror >= bins[b]) & (r_mirror < bins[b+1])).sum()

C_std    = N_std    / N_total
C_mirror = N_mirror / N_total
C_excess = C_mirror - C_std

peak_r = bin_centers[np.argmax(C_excess)]
print('Mirror excess peak at: r =', peak_r, 'Mpc')
print('Predicted peak at:     r =', r_pred, 'Mpc')

In [ ]:
# CELL 5 -- Monte Carlo significance
N_MC = 300
print('Monte Carlo null: randomising mirror axis (N=' + str(N_MC) + ')...')

mc_excess = np.zeros((N_MC, N_BINS))
rng_mc = np.random.default_rng(99)

for i in range(N_MC):
    # Random axis
    theta = rng_mc.uniform(0, np.pi)
    phi   = rng_mc.uniform(0, 2*np.pi)
    n_rand = np.array([np.sin(theta)*np.cos(phi),
                       np.sin(theta)*np.sin(phi),
                       np.cos(theta)])
    proj_r = pos @ n_rand
    pos_rand_mirror = pos - 2 * np.outer(proj_r, n_rand)
    r_m = np.linalg.norm(pos_rand_mirror[idx1] - pos[idx2], axis=1)
    N_m = np.zeros(N_BINS)
    for b in range(N_BINS):
        N_m[b] = ((r_m >= bins[b]) & (r_m < bins[b+1])).sum()
    mc_excess[i] = N_m/N_total - C_std
    if (i+1) % 75 == 0:
        print('  MC', i+1, '/', N_MC)

mc_mean = mc_excess.mean(axis=0)
mc_std  = mc_excess.std(axis=0)
sigma   = (C_excess - mc_mean) / (mc_std + 1e-10)

pred_bin   = np.argmin(np.abs(bin_centers - r_pred))
sigma_pred = sigma[pred_bin]
print('Sigma at r =', r_pred, 'Mpc: sigma =', round(sigma_pred, 2))
print('Max sigma:', round(sigma.max(), 2), 'at r =', bin_centers[sigma.argmax()], 'Mpc')

In [ ]:
# CELL 6 -- Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

ax = axes[0]
ax.fill_between(bin_centers, mc_mean-2*mc_std, mc_mean+2*mc_std,
                alpha=0.3, color='gray', label='2sigma null')
ax.plot(bin_centers, C_excess, color='#42A5F5', lw=2, label='Mirror excess')
ax.axvline(r_pred, color='yellow', lw=2, ls='--', label='T1/2=830 Mpc')
ax.axhline(0, color='white', lw=0.5, alpha=0.4)
ax.set_xlabel('r [Mpc]', color='white')
ax.set_ylabel('C_mirror - C_std', color='white')
ax.set_title('P1: Mirror Correlation (KTF3-axis corrected)\nBOSS DR12', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

ax = axes[1]
ax.plot(bin_centers, sigma, color='#42A5F5', lw=2)
ax.axhline(0,  color='white',  lw=0.5, alpha=0.4)
ax.axhline(2,  color='orange', lw=1.5, ls='--', alpha=0.7, label='2sigma')
ax.axhline(-2, color='orange', lw=1.5, ls='--', alpha=0.7)
ax.axvline(r_pred, color='yellow', lw=2, ls='--', label='Predicted: 830 Mpc')
ax.set_xlabel('r [Mpc]', color='white')
ax.set_ylabel('Significance sigma', color='white')
ax.set_title('P1 Significance (axis-corrected)\nsigma at 830 Mpc = ' + str(round(sigma_pred,2)), color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

plt.suptitle('v48b: KTF3 P1 -- BOSS DR12 -- KTF3-axis corrected -- Cotting (2026)', color='white', fontsize=12)
plt.tight_layout()
plt.savefig('v48b_boss_mirror_corrected.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# CELL 7 -- Summary
print('\n' + '='*60)
print('RESULT v48b: KTF3 P1 MIRROR CORRELATION -- BOSS DR12')
print('AXIS-CORRECTED (galactic frame, KTF3 axis l=277, b=-31)')
print('Cotting, S. (2026) -- PRE-REGISTERED March 2026')
print('='*60)
print('  Galaxies:', N_gal)
print('  Mirror axis: l =', L_KTF3, 'b =', B_KTF3)
print()
print('  Predicted peak: r = 830 Mpc')
print('  Observed peak:  r =', peak_r, 'Mpc')
print('  Sigma at r=830: ', round(sigma_pred, 2))
print()
if sigma_pred > 2:
    verdict = 'CONFIRMED -- Peak at predicted location.'
elif sigma_pred > 1:
    verdict = 'MARGINAL -- Weak excess.'
elif sigma_pred > 0:
    verdict = 'WEAK -- Correct sign, not significant.'
else:
    verdict = 'NULL -- No mirror excess at predicted location.'
print('  Verdict:', verdict)
print()
print('  NOTE: BOSS covers ~10,000 deg2 (25% sky).')
print('  Partial sky coverage creates geometric systematics.')
print('  Euclid DR1 (36% sky, uniform) will be definitive.')
print()
print('  Comparison with v48 (wrong axis):')
print('  v48  sigma at 830 Mpc = -1.46  (RA=0 axis -- WRONG)')
print('  v48b sigma at 830 Mpc =', round(sigma_pred, 2), ' (KTF3 axis -- CORRECT)')
print()
print('  Pre-registration: academia.edu/AndrewCotting')
print('  GitHub: github.com/Andrew-Cot/KTF3-Analyse')
print('='*60)